# Systems of Equations to Matrices (Row Echelon Form)

*Course notes for **Math for Machine Learning**, C1 · W2 · L1 — "Systems of Equations to Matrices" (DeepLearning.AI).*

Elimination only ever touches the **coefficients**, so we can drop the variables and operate on the matrix directly. The same row operations that solved the system now carry the matrix through two milestone shapes:

$$ \text{matrix} \;\xrightarrow{\text{eliminate down}}\; \textbf{row echelon form (REF)} \;\xrightarrow{\text{eliminate up}}\; \textbf{reduced row echelon form (RREF)}. $$

In [1]:
import numpy as np
from fractions import Fraction as F

def to_F(M):
    return [[F(x) for x in row] for row in M]

def show(M):
    for row in M:
        print("  [" + "  ".join(f"{str(x):>5}" for x in row) + "]")

In [2]:
def row_echelon(M):
    """Forward elimination to row echelon form (leading 1s, zeros below)."""
    M = [row[:] for row in to_F(M)]
    rows, cols = len(M), len(M[0])
    pivot_row = 0
    for col in range(cols):
        # find a row at/below pivot_row with a non-zero entry in this column
        piv = next((r for r in range(pivot_row, rows) if M[r][col] != 0), None)
        if piv is None:
            continue
        M[pivot_row], M[piv] = M[piv], M[pivot_row]          # swap up
        M[pivot_row] = [x / M[pivot_row][col] for x in M[pivot_row]]  # scale pivot to 1
        for r in range(pivot_row + 1, rows):                  # clear below
            M[r] = [M[r][k] - M[r][col] * M[pivot_row][k] for k in range(cols)]
        pivot_row += 1
    return M

def reduced_row_echelon(M):
    """Continue from REF, clearing entries ABOVE each pivot (Gauss-Jordan)."""
    M = row_echelon(M)
    rows, cols = len(M), len(M[0])
    for r in range(rows - 1, -1, -1):
        piv = next((c for c in range(cols) if M[r][c] != 0), None)
        if piv is None:
            continue
        for r2 in range(r):                                   # clear above
            M[r2] = [M[r2][k] - M[r2][piv] * M[r][k] for k in range(cols)]
    return M

## 1. Non-singular example

$$ \begin{cases} 5a + b = 17 \\ 4a - 3b = 6 \end{cases} \;\longrightarrow\; \text{coefficient matrix } \begin{bmatrix} 5 & 1 \\ 4 & -3 \end{bmatrix} $$

Eliminating reproduces the solving steps from earlier lectures:

$$ \begin{bmatrix} 5 & 1 \\ 4 & -3 \end{bmatrix} \xrightarrow{\text{REF}} \begin{bmatrix} 1 & 0.2 \\ 0 & 1 \end{bmatrix} \xrightarrow{\text{RREF}} \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} $$

The RREF of a **non-singular** matrix is always the **identity** — the algebraic fingerprint of a unique solution.

In [3]:
A = [[5, 1], [4, -3]]
print("Original:");           show(A)
print("\nRow echelon form (REF):"); show(row_echelon(A))
print("\nReduced REF (RREF):");     show(reduced_row_echelon(A))

Original:
  [    5      1]
  [    4     -3]

Row echelon form (REF):
  [    1    1/5]
  [    0      1]

Reduced REF (RREF):
  [    1      0]
  [    0      1]


## 2. Singular examples

A **redundant** system $\begin{cases}a+b=10\\2a+2b=20\end{cases}$ has matrix $\begin{bmatrix}1&1\\2&2\end{bmatrix}$. Elimination produces a **zero row** — the REF is $\begin{bmatrix}1&1\\0&0\end{bmatrix}$, which is *not* the identity. A fully degenerate system ($0=0$ twice) is the zero matrix and stays zero.

A zero row in the REF is exactly the signal that the matrix is **singular** (fewer pivots than rows).

In [4]:
for name, M in {"redundant [[1,1],[2,2]]": [[1, 1], [2, 2]],
                "zero      [[0,0],[0,0]]": [[0, 0], [0, 0]]}.items():
    ref = row_echelon(M)
    pivots = sum(any(x != 0 for x in row) for row in ref)
    print(name, "-> REF:")
    show(ref)
    print(f"   pivots = {pivots}/{len(M)}  ->  {'non-singular' if pivots == len(M) else 'singular'}\n")

redundant [[1,1],[2,2]] -> REF:
  [    1      1]
  [    0      0]
   pivots = 1/2  ->  singular

zero      [[0,0],[0,0]] -> REF:
  [    0      0]
  [    0      0]
   pivots = 0/2  ->  singular



## 3. What "row echelon form" means

A matrix is in **row echelon form** when it forms a descending **staircase**:

- Each non-zero row starts with a **leading 1** (a *pivot*).
- Every pivot sits strictly to the **right** of the pivot in the row above.
- Rows of all zeros sit at the **bottom**.
- All entries **below** a pivot are 0 (`*` above/right may be anything).

$$
\begin{bmatrix} 1 & * & * & * \\ 0 & 1 & * & * \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 0 & 0 \end{bmatrix}
$$

**Reduced** row echelon form adds one more rule: each pivot is the *only* non-zero entry in its column (entries **above** pivots are cleared to 0 as well). For a non-singular square matrix that forces the **identity**.

Valid 2×2 REF shapes: $\begin{bmatrix}1&*\\0&1\end{bmatrix}$ (non-singular), $\begin{bmatrix}1&*\\0&0\end{bmatrix}$ and $\begin{bmatrix}0&0\\0&0\end{bmatrix}$ (singular).

## 4. Quiz

$$ \begin{cases} 5a + b = 11 \\ 10a + 2b = 22 \end{cases} \;\longrightarrow\; \begin{bmatrix} 5 & 1 \\ 10 & 2 \end{bmatrix} $$

Row 2 is $2\times$ row 1, so elimination yields a zero row → REF $\begin{bmatrix}1&0.2\\0&0\end{bmatrix}$ → **singular** (redundant, infinitely many solutions).

In [5]:
Q = [[5, 1], [10, 2]]
ref = row_echelon(Q)
print("REF:"); show(ref)
pivots = sum(any(x != 0 for x in row) for row in ref)
print(f"\npivots = {pivots}/2  ->  {'non-singular' if pivots == 2 else 'singular (redundant)'}")

REF:
  [    1    1/5]
  [    0      0]

pivots = 1/2  ->  singular (redundant)


## 5. Takeaways

- A system's coefficients form a **matrix**; elimination is just **row operations** on it.
- **Row echelon form (REF):** staircase of leading 1s, zeros below them — reached by forward elimination.
- **Reduced REF (RREF):** also zeros *above* each pivot — reached by continuing (Gauss–Jordan).
- **Non-singular** ⇔ a pivot in every row ⇔ RREF is the **identity** ⇔ unique solution.
- **Singular** ⇔ at least one **zero row** appears in the REF (fewer pivots than rows).

*Companion:* [solving with more variables](./C1_W2_L1_solving_systems_more_variables.ipynb) shows the same elimination written out as equations.